# Gradient Boosting Machines y XGBoost
### Curso de Aprendizaje de Máquina — Actuaría

---
## 1. Fundamentos teóricos: Boosting como descenso en gradiente funcional

Sea $(x_i, y_i)_{i=1}^n$ el conjunto de entrenamiento. El objetivo de boosting es encontrar una función aditiva
$$F_M(x) = \sum_{m=0}^{M} \nu_m h_m(x)$$
donde cada $h_m$ es un *aprendiz débil* (árbol de decisión poco profundo) y $\nu_m$ es la tasa de aprendizaje.

### 1.1 AdaBoost como preámbulo
AdaBoost (Freund & Schapire 1997) repondera muestras mal clasificadas exponencialmente:
$$w_i^{(m+1)} = w_i^{(m)} \exp(-y_i \alpha_m h_m(x_i)), \quad \alpha_m = \frac{1}{2}\ln\frac{1-\varepsilon_m}{\varepsilon_m}$$
Friedman (2001) demostró que AdaBoost equivale a minimizar la pérdida exponencial $L(y,F)=e^{-yF}$ por descenso en gradiente **funcional**.

### 1.2 Gradient Boosting genérico (Friedman 2001)
Para una pérdida diferenciable $\mathcal{L}$ arbitraria, el algoritmo es:

**Algoritmo GBM:**
1. Inicializar $F_0(x) = \arg\min_\gamma \sum_i L(y_i, \gamma)$
2. Para $m = 1, \ldots, M$:
   - Calcular pseudo-residuales: $r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$
   - Ajustar $h_m$ a los pseudo-residuales $(x_i, r_{im})$
   - Buscar paso óptimo: $\rho_m = \arg\min_\rho \sum_i L(y_i, F_{m-1}(x_i) + \rho h_m(x_i))$
   - Actualizar: $F_m(x) = F_{m-1}(x) + \nu \cdot \rho_m h_m(x)$

**Pérdidas comunes y sus gradientes:**

| Tarea | $L(y,F)$ | $r_i = -\partial_F L$ |
|---|---|---|
| Regresión (MSE) | $\frac{1}{2}(y-F)^2$ | $y - F$ |
| Regresión (MAE) | $|y-F|$ | $\text{sgn}(y-F)$ |
| Clasificación binaria | $\log(1+e^{-yF})$ | $y\sigma(-yF)$ |
| Poisson (count) | $-y F + e^F$ | $y - e^F$ |

### 1.3 Regularización en GBM
- **Shrinkage** (tasa de aprendizaje $\nu \in (0,1]$): reduce cada árbol, requiere más árboles pero generaliza mejor.
- **Subsampling estocástico** (Friedman 2002): muestrear fracción $\eta$ de filas por iteración → SGD funcional.
- **Profundidad máxima** de árboles: controla la complejidad de interacciones capturadas ($d=1$ → modelo aditivo puro).


## 2. XGBoost: la segunda derivada importa

Chen & Guestrin (2016) reescriben el objetivo de cada paso $m$ con expansión de Taylor de **segundo orden**:
$$\mathcal{L}^{(m)} = \sum_{i=1}^n \left[g_i h(x_i) + \frac{1}{2} h_i h(x_i)^2\right] + \Omega(h)$$
donde $g_i = \partial_F L(y_i, F_{m-1})$, $h_i = \partial_F^2 L(y_i, F_{m-1})$, y la regularización es:
$$\Omega(h) = \gamma T + \frac{1}{2}\lambda \|w\|^2$$
con $T$ = número de hojas y $w_j$ = valor de la hoja $j$.

Para hoja $j$ (con instancias $I_j$), la solución analítica es:
$$w_j^* = -\frac{\sum_{i\in I_j} g_i}{\sum_{i\in I_j} h_i + \lambda}$$
y el **gain** de un split es:
$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

**Ventajas sobre GBM clásico:**
- La hessiana $h_i$ pondera muestras automáticamente (efecto de las muestras difíciles).
- El gain formula cuánto vale el split → poda exacta con `min_child_weight`.
- Algoritmos de split aproximados (quantile sketch) → escala a millones de filas.
- Regularización $L_1$ (`alpha`) y $L_2$ (`lambda`) nativa.
- Column subsampling por árbol, nivel o split.


## 3. GBM desde cero (sin librerías)
Implementamos GBM con pérdida MSE y árboles de regresión mediante búsqueda exhaustiva de splits.

In [ ]:
import numpy as np

class StumpRegressor:
    """Árbol de regresión de profundidad máxima `max_depth`."""
    def __init__(self, max_depth=3, min_samples_leaf=5):
        self.max_depth=max_depth; self.min_samples_leaf=min_samples_leaf
        self.tree=None

    def _best_split(self, X, r):
        best={'gain':-np.inf,'feat':None,'thr':None}
        var_total=np.var(r)*len(r)
        for j in range(X.shape[1]):
            vals=np.unique(X[:,j])
            for thr in (vals[:-1]+vals[1:])/2:
                l=r[X[:,j]<=thr]; r_=r[X[:,j]>thr]
                if len(l)<self.min_samples_leaf or len(r_)<self.min_samples_leaf: continue
                gain=var_total - (np.var(l)*len(l)+np.var(r_)*len(r_))
                if gain>best['gain']: best={'gain':gain,'feat':j,'thr':thr}
        return best

    def _build(self, X, r, depth):
        if depth==0 or len(r)<2*self.min_samples_leaf:
            return {'leaf':True,'val':np.mean(r)}
        split=self._best_split(X,r)
        if split['feat'] is None: return {'leaf':True,'val':np.mean(r)}
        mask=X[:,split['feat']]<=split['thr']
        return {'leaf':False,'feat':split['feat'],'thr':split['thr'],
                'left':self._build(X[mask],r[mask],depth-1),
                'right':self._build(X[~mask],r[~mask],depth-1)}

    def fit(self, X, r): self.tree=self._build(X,r,self.max_depth); return self

    def _predict_row(self, node, x):
        if node['leaf']: return node['val']
        return self._predict_row(node['left'] if x[node['feat']]<=node['thr'] else node['right'], x)

    def predict(self, X): return np.array([self._predict_row(self.tree,x) for x in X])


class GBMScratch:
    """
    GBM con pérdida MSE desde cero.
    Soporta subsampling estocástico por iteración.
    """
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3,
                 min_samples_leaf=5, subsample=1.0, random_state=42):
        self.M=n_estimators; self.nu=learning_rate; self.d=max_depth
        self.msl=min_samples_leaf; self.sub=subsample; self.rs=random_state
        self.trees=[]; self.F0=None

    def fit(self, X, y):
        rng=np.random.default_rng(self.rs)
        self.F0=np.mean(y)       # init: minimiza MSE global
        F=np.full(len(y), self.F0)
        for _ in range(self.M):
            r=y-F                # pseudo-residuales MSE = gradiente negativo
            if self.sub<1.0:
                idx=rng.choice(len(y), int(len(y)*self.sub), replace=False)
                h=StumpRegressor(self.d, self.msl).fit(X[idx], r[idx])
            else:
                h=StumpRegressor(self.d, self.msl).fit(X, r)
            self.trees.append(h)
            F+=self.nu*h.predict(X)
        return self

    def predict(self, X):
        return self.F0 + self.nu*sum(h.predict(X) for h in self.trees)

    def staged_predict(self, X):
        """Generador: predicción acumulada en cada etapa."""
        F=np.full(len(X), self.F0)
        for h in self.trees:
            F=F+self.nu*h.predict(X)
            yield F.copy()

## 4. Ejemplo 1 — Regresión: datos California Housing (muestra)
Comparamos GBM desde cero vs `GradientBoostingRegressor` de scikit-learn vs `XGBRegressor`.
Iteramos un grid de hiperparámetros y seleccionamos con λ-score que penaliza overfitting:
$$\lambda\text{-score} = R^2_{\text{test}} - \alpha\cdot|R^2_{\text{train}} - R^2_{\text{test}}|$$

In [ ]:
import pandas as pd, numpy as np, warnings, gc
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
warnings.filterwarnings('ignore')

X,y = fetch_california_housing(return_X_y=True)
X,y = X[:5000], y[:5000]  # muestra rápida
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.2,random_state=42)
sc=StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xte_s=sc.transform(Xte)

def lambda_score(r2_tr, r2_te, alpha=0.5):
    return r2_te - alpha*abs(r2_tr-r2_te)

# ── GBM desde cero ──────────────────────────────────────────────
grid_scratch = [
    {'n_estimators':n,'learning_rate':lr,'max_depth':d,'subsample':s}
    for n in [50,100,200]
    for lr in [0.05,0.1]
    for d in [2,3]
    for s in [1.0,0.7]
]
rows=[]
for p in grid_scratch:
    m=GBMScratch(**p,random_state=42).fit(Xtr_s,ytr)
    r2tr=r2_score(ytr,m.predict(Xtr_s)); r2te=r2_score(yte,m.predict(Xte_s))
    rows.append({**p,'R2_train':round(r2tr,4),'R2_test':round(r2te,4),'λ':round(lambda_score(r2tr,r2te),4)})
    del m; gc.collect()

df_scratch=pd.DataFrame(rows).sort_values('λ',ascending=False)
print('=== GBM Scratch — top 5 ==='); print(df_scratch.head())

In [ ]:
# ── sklearn GradientBoostingRegressor ───────────────────────────
grid_sk = [
    {'n_estimators':n,'learning_rate':lr,'max_depth':d,'subsample':s,'loss':loss}
    for n in [100,300]
    for lr in [0.05,0.1]
    for d in [3,4]
    for s in [1.0,0.7]
    for loss in ['squared_error','huber']
]
rows=[]
for p in grid_sk:
    m=GradientBoostingRegressor(**p,random_state=42).fit(Xtr_s,ytr)
    r2tr=r2_score(ytr,m.predict(Xtr_s)); r2te=r2_score(yte,m.predict(Xte_s))
    rows.append({**p,'R2_train':round(r2tr,4),'R2_test':round(r2te,4),'λ':round(lambda_score(r2tr,r2te),4)})
    del m; gc.collect()

df_sk=pd.DataFrame(rows).sort_values('λ',ascending=False)
print('=== sklearn GBM — top 5 ==='); print(df_sk.head())

In [ ]:
# ── XGBoost ────────────────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    xgb_available=True
except ImportError:
    print('pip install xgboost'); xgb_available=False

if xgb_available:
    grid_xgb = [
        {'n_estimators':n,'learning_rate':lr,'max_depth':d,'subsample':s,
         'colsample_bytree':cs,'reg_lambda':lam,'reg_alpha':alp}
        for n in [200,500]
        for lr in [0.05,0.1]
        for d in [3,4]
        for s in [0.8,1.0]
        for cs in [0.8,1.0]
        for lam in [1,5]
        for alp in [0,0.1]
    ]
    rows=[]
    for p in grid_xgb:
        m=XGBRegressor(**p,tree_method='hist',random_state=42,verbosity=0).fit(Xtr_s,ytr)
        r2tr=r2_score(ytr,m.predict(Xtr_s)); r2te=r2_score(yte,m.predict(Xte_s))
        rows.append({**p,'R2_train':round(r2tr,4),'R2_test':round(r2te,4),'λ':round(lambda_score(r2tr,r2te),4)})
        del m; gc.collect()
    df_xgb=pd.DataFrame(rows).sort_values('λ',ascending=False)
    print('=== XGBoost — top 5 ==='); print(df_xgb.head())

In [ ]:
import matplotlib.pyplot as plt

# Tabla comparativa de ganadores
best_scratch=df_scratch.iloc[0]
best_sk=df_sk.iloc[0]

fig,axes=plt.subplots(1,2,figsize=(13,4))

# ── staged R² del ganador sklearn ──
bp=best_sk.drop(['R2_train','R2_test','λ']).to_dict()
bp={k:v for k,v in bp.items() if k in ['n_estimators','learning_rate','max_depth','subsample','loss']}
m_best=GradientBoostingRegressor(**bp,random_state=42).fit(Xtr_s,ytr)
tr_staged=[r2_score(ytr,p) for p in m_best.staged_predict(Xtr_s)]
te_staged=[r2_score(yte,p) for p in m_best.staged_predict(Xte_s)]
ax=axes[0]; ax.plot(tr_staged,label='train'); ax.plot(te_staged,label='test')
ax.set_title('sklearn GBM — R² por iteración',fontsize=11); ax.set_xlabel('n_estimators'); ax.set_ylabel('R²')
ax.legend(); ax.grid(alpha=.3)
del m_best; gc.collect()

# ── GBM scratch staged ──
sp=dict(n_estimators=int(best_scratch.n_estimators),learning_rate=best_scratch.learning_rate,
        max_depth=int(best_scratch.max_depth),subsample=best_scratch.subsample)
m_sc=GBMScratch(**sp,random_state=42).fit(Xtr_s,ytr)
tr_st=[r2_score(ytr,p) for p in m_sc.staged_predict(Xtr_s)]
te_st=[r2_score(yte,p) for p in m_sc.staged_predict(Xte_s)]
ax=axes[1]; ax.plot(tr_st,label='train'); ax.plot(te_st,label='test')
ax.set_title('GBM Scratch — R² por iteración',fontsize=11); ax.set_xlabel('n_estimators'); ax.set_ylabel('R²')
ax.legend(); ax.grid(alpha=.3)
del m_sc; gc.collect()

plt.tight_layout(); plt.savefig('staged_r2.png',dpi=100,bbox_inches='tight'); plt.show()

## 5. Ejemplo 2 — Clasificación binaria: Pima Diabetes
Usamos la función logística (log-loss). Para GBM, las probabilidades salen de:
$$P(y=1|x) = \sigma(F_M(x)) = \frac{1}{1+e^{-F_M(x)}}$$
Los pseudo-residuales son $r_i = y_i - \sigma(F_{m-1}(x_i))$.

In [ ]:
import pandas as pd, numpy as np, gc
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.ensemble import GradientBoostingClassifier

url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols=['preg','gluc','bp','skin','ins','bmi','dpf','age','target']
df=pd.read_csv(url,names=cols)
X,y=df.drop('target',axis=1).values, df['target'].values
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
sc=StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xte_s=sc.transform(Xte)

def lambda_score_cls(auc_tr, auc_te, alpha=0.5):
    return auc_te - alpha*abs(auc_tr-auc_te)

grid_cls=[
    {'n_estimators':n,'learning_rate':lr,'max_depth':d,'subsample':s,'min_samples_leaf':msl}
    for n in [100,300]
    for lr in [0.05,0.1]
    for d in [2,3,4]
    for s in [0.7,1.0]
    for msl in [5,20]
]
rows=[]
for p in grid_cls:
    m=GradientBoostingClassifier(**p,random_state=42).fit(Xtr_s,ytr)
    atr=roc_auc_score(ytr,m.predict_proba(Xtr_s)[:,1])
    ate=roc_auc_score(yte,m.predict_proba(Xte_s)[:,1])
    rows.append({**p,'AUC_train':round(atr,4),'AUC_test':round(ate,4),'λ':round(lambda_score_cls(atr,ate),4)})
    del m; gc.collect()

df_cls=pd.DataFrame(rows).sort_values('λ',ascending=False)
print('=== GBM Clasificación — top 5 ==='); print(df_cls.head())

In [ ]:
# XGBoost clasificación
if xgb_available:
    from xgboost import XGBClassifier
    grid_xgb_cls=[
        {'n_estimators':n,'learning_rate':lr,'max_depth':d,'subsample':s,
         'colsample_bytree':cs,'reg_lambda':lam,'scale_pos_weight':spw}
        for n in [200,500]
        for lr in [0.05,0.1]
        for d in [3,4]
        for s in [0.8,1.0]
        for cs in [0.8,1.0]
        for lam in [1,5]
        for spw in [1.0, sum(ytr==0)/sum(ytr==1)]  # balance de clases
    ]
    rows=[]
    for p in grid_xgb_cls:
        m=XGBClassifier(**p,use_label_encoder=False,eval_metric='logloss',
                        tree_method='hist',random_state=42,verbosity=0).fit(Xtr_s,ytr)
        atr=roc_auc_score(ytr,m.predict_proba(Xtr_s)[:,1])
        ate=roc_auc_score(yte,m.predict_proba(Xte_s)[:,1])
        rows.append({**p,'AUC_train':round(atr,4),'AUC_test':round(ate,4),'λ':round(lambda_score_cls(atr,ate),4)})
        del m; gc.collect()
    df_xgb_cls=pd.DataFrame(rows).sort_values('λ',ascending=False)
    print('=== XGBoost Clasificación — top 5 ==='); print(df_xgb_cls.head())

## 6. Ejemplo 3 — Regresión de conteos: Poisson (dataset de accidentes)
Para datos de conteo $y \in \mathbb{N}_0$, la pérdida Poisson es:
$$L(y, F) = e^F - y\cdot F$$
con gradiente $g_i = e^{F_i} - y_i$ y hessiana $h_i = e^{F_i}$ (XGBoost los usa directamente).
La predicción es $\hat{y} = e^{F_M(x)}$.

In [ ]:
import numpy as np, gc
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_poisson_deviance
from sklearn.ensemble import GradientBoostingRegressor

# Dataset: frecuencia de siniestros MTPL (French Motor)
mtpl=fetch_openml('freMTPL2freq',data_home='/tmp',as_frame=True,parser='auto')
dfm=mtpl.frame.dropna().sample(20000,random_state=42)

# features numéricas relevantes + target
num_feats=['Exposure','VehPower','VehAge','DrivAge','BonusMalus','Density']
dfm=dfm[num_feats+['ClaimNb']].copy()
dfm['ClaimNb']=dfm['ClaimNb'].astype(int)
X,y=dfm[num_feats].values, dfm['ClaimNb'].values
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
sc=StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xte_s=sc.transform(Xte)

def poisson_lambda(dev_tr, dev_te, alpha=0.5):
    # menor deviance = mejor; invertimos signo
    return -dev_te - alpha*abs(dev_tr-dev_te)

grid_poi=[
    {'n_estimators':n,'learning_rate':lr,'max_depth':d,'subsample':s}
    for n in [100,300]
    for lr in [0.05,0.1]
    for d in [3,4]
    for s in [0.7,1.0]
]
rows=[]
for p in grid_poi:
    m=GradientBoostingRegressor(**p,loss='poisson',random_state=42).fit(Xtr_s,ytr)
    ptr=m.predict(Xtr_s).clip(1e-6)
    pte=m.predict(Xte_s).clip(1e-6)
    dtr=mean_poisson_deviance(ytr,ptr); dte=mean_poisson_deviance(yte,pte)
    rows.append({**p,'Dev_train':round(dtr,4),'Dev_test':round(dte,4),'λ':round(poisson_lambda(dtr,dte),4)})
    del m; gc.collect()

df_poi=pd.DataFrame(rows).sort_values('λ',ascending=False)
print('=== GBM Poisson — top 5 ==='); print(df_poi.head())

## 7. Importancia de características y SHAP
XGBoost provee tres tipos de importancia:
- **weight**: número de splits que usan la feature.
- **gain**: reducción promedio de pérdida al usar la feature (más interpretable).
- **cover**: número de muestras promedio cubierto por splits de esa feature.

SHAP (Lundberg & Lee 2017) descompone $F(x) = \phi_0 + \sum_j \phi_j$ con garantías de equidad (Shapley values exactos en O(TLD) para árboles).

In [ ]:
import matplotlib.pyplot as plt, gc
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

if xgb_available:
    from xgboost import XGBRegressor, plot_importance
    X,y=fetch_california_housing(return_X_y=True); X,y=X[:5000],y[:5000]
    fnames=fetch_california_housing().feature_names
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
    sc=StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xte_s=sc.transform(Xte)

    # ganador de grid_xgb (hiperparámetros del top-1)
    best=df_xgb.iloc[0].drop(['R2_train','R2_test','λ']).to_dict()
    best={k:v for k,v in best.items() if k in ['n_estimators','learning_rate','max_depth','subsample','colsample_bytree','reg_lambda','reg_alpha']}
    best['n_estimators']=int(best['n_estimators']); best['max_depth']=int(best['max_depth'])
    m_final=XGBRegressor(**best,tree_method='hist',random_state=42,verbosity=0)
    m_final.fit(Xtr_s,ytr,feature_names=fnames)

    fig,axes=plt.subplots(1,2,figsize=(13,4))
    for ax,imp in zip(axes,['gain','cover']):
        plot_importance(m_final,ax=ax,importance_type=imp,max_num_features=8,
                        title=f'Importancia por {imp}',show_values=False)
    plt.tight_layout(); plt.savefig('feat_importance.png',dpi=100,bbox_inches='tight'); plt.show()

    # SHAP (si está instalado)
    try:
        import shap
        explainer=shap.TreeExplainer(m_final)
        sv=explainer.shap_values(Xte_s[:200])
        shap.summary_plot(sv,Xte_s[:200],feature_names=fnames,show=False,plot_size=(9,4))
        plt.tight_layout(); plt.savefig('shap.png',dpi=100,bbox_inches='tight'); plt.show()
        del explainer,sv; gc.collect()
    except ImportError:
        print('pip install shap')
    del m_final; gc.collect()

## 8. Early stopping con validación cruzada
XGBoost permite early stopping nativo. Aquí lo combinamos con CV estratificada para clasificación.

In [ ]:
import numpy as np, gc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

if xgb_available:
    from xgboost import XGBClassifier
    # re-cargamos Pima
    import pandas as pd
    url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
    cols=['preg','gluc','bp','skin','ins','bmi','dpf','age','target']
    df=pd.read_csv(url,names=cols)
    X,y=df.drop('target',axis=1).values,df['target'].values

    cv=StratifiedKFold(5,shuffle=True,random_state=42)
    aucs=[]
    for tr_i,te_i in cv.split(X,y):
        Xtr_,Xte_=X[tr_i],X[te_i]; ytr_,yte_=y[tr_i],y[te_i]
        m=XGBClassifier(n_estimators=1000,learning_rate=0.05,max_depth=3,
                        subsample=0.8,colsample_bytree=0.8,reg_lambda=1,
                        tree_method='hist',random_state=42,verbosity=0,
                        early_stopping_rounds=30,eval_metric='logloss')
        m.fit(Xtr_,ytr_,eval_set=[(Xte_,yte_)],verbose=False)
        aucs.append(roc_auc_score(yte_,m.predict_proba(Xte_)[:,1]))
        print(f'  fold AUC={aucs[-1]:.4f}  best_iter={m.best_iteration}')
        del m; gc.collect()
    print(f'\nCV AUC = {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

## 9. Resumen de hiperparámetros clave

| Parámetro | GBM sklearn | XGBoost | Efecto |
|---|---|---|---|
| `n_estimators` / `n_estimators` | ✓ | ✓ | más árboles → más capacidad, riesgo de overfitting |
| `learning_rate` | ✓ | `learning_rate` | shrinkage: ↓ → requiere más árboles, generaliza mejor |
| `max_depth` | ✓ | ✓ | profundidad máxima: controla complejidad de interacciones |
| `subsample` | ✓ | ✓ | fracción de filas por árbol: regularización estocástica |
| `min_samples_leaf` | ✓ | `min_child_weight` | mínimo de muestras/suma hessiana en hoja |
| — | — | `colsample_bytree` | fracción de columnas por árbol |
| — | — | `reg_lambda` | regularización L2 en pesos de hojas |
| — | — | `reg_alpha` | regularización L1 en pesos de hojas |
| — | — | `gamma` | ganancia mínima para hacer un split |
| — | — | `scale_pos_weight` | balance de clases en clasificación binaria |

**Regla práctica:** ajustar en este orden: `max_depth` + `min_child_weight` → `subsample` + `colsample_bytree` → `reg_lambda` + `reg_alpha` → `learning_rate` (reducir) + `n_estimators` (aumentar).